<a href="https://colab.research.google.com/github/kimdonggyu2008/Personal_Study/blob/main/conformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# %cd /content/drive/MyDrive/코딩공부/project_folder

/content/drive/MyDrive/코딩공부/project_folder


In [4]:
# !git clone https://github.com/sooftware/conformer.git

Cloning into 'conformer'...
remote: Enumerating objects: 474, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 474 (delta 40), reused 30 (delta 28), pack-reused 405 (from 1)
Receiving objects: 100% (474/474), 2.88 MiB | 12.84 MiB/s, done.
Resolving deltas: 100% (266/266), done.


# utils.py

In [ ]:
from abc import ABC, abstractmethod
from typing import Union, Tuple
from pathlib import Path
from torch import Tensor
import torchaudio
import json


def save_json(file_path: str, data: dict):
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f)


class IPipeline(ABC):
    @abstractmethod
    def run():
        """Used to run all the callables functions sequantially
        """
        pass


def load_audio(file_path: Union[str, Path]) -> Tuple[Tensor, int]:
    x, sr = torchaudio.load(file_path, normalize=True)
    return x, sr

# tokenizer


In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod, abstractproperty
from dataclasses import dataclass
from typing import (
    Callable,
    List,
    Tuple,
    Union
    )
from os import PathLike
from data_loaders import JSONLoader
from utils import save_json
from functools import wraps

PAD = '<PAD>'
SOS = '<SOS>'
EOS = '<EOS>'
BLANK = '<BLANK>'


def check_token(token: str) -> Callable:
    """To check if a token exists or not

    Args:
        token ([type]): the token to be checked
    """
    def decorator(func):
        @wraps(func)
        def wrapper(obj, token=token):
            if token in obj._token_to_id:
                return obj._token_to_id[token]
            return func(obj, token)
        return wrapper
    return decorator


@dataclass
class SpecialTokens:
    _pad: Tuple[str, int] = (None, None)
    _blank: Tuple[str, int] = (None, None)
    _sos: Tuple[str, int] = (None, None)
    _eos: Tuple[str, int] = (None, None)

    @property
    def pad_id(self):
        return self._pad[1]

    @property
    def pad_token(self):
        return self._pad[0]

    @property
    def blank_id(self):
        return self._blank[1]

    @property
    def blank_token(self):
        return self._blank[0]

    @property
    def sos_id(self):
        return self._sos[1]

    @property
    def sos_token(self):
        return self._sos[0]

    @property
    def eos_id(self):
        return self._eos[1]

    @property
    def eos_token(self):
        return self._eos[0]

    @property
    def mask_id(self):
        return self._mask[1]

    @property
    def mask_token(self):
        return self._mask[0]


class ITokenizer(ABC):

    @abstractmethod
    def ids2tokens(self):
        pass

    @abstractmethod
    def tokens2ids(self):
        pass

    @abstractmethod
    def set_tokenizer(self):
        pass

    @abstractmethod
    def save_tokenizer(self):
        pass

    @abstractmethod
    def load_tokenizer(self):
        pass

    @abstractmethod
    def add_token(self):
        pass

    @abstractmethod
    def preprocess_tokens(self):
        pass

    @abstractmethod
    def batch_tokenizer(self):
        pass

    @abstractproperty
    def vocab_size(self):
        pass

    @abstractmethod
    def get_tokens(self):
        pass


class BaseTokenizer(ITokenizer):
    _pad_key = 'pad'
    _sos_key = 'sos'
    _eos_key = 'eos'
    _blank_key = 'blank'
    _token_to_id_key = 'token_to_id'
    _special_tokens_key = 'special_tokens'

    def __init__(self) -> None:
        super().__init__()
        self._token_to_id = dict()
        self._id_to_token = dict()
        self.special_tokens = SpecialTokens()

    @property
    def vocab_size(self):
        return len(self._token_to_id)

    def add_token(self, token: str):
        token_id = self.vocab_size
        self._token_to_id[token] = token_id
        self._id_to_token[token_id] = token
        return token_id

    @check_token(PAD)
    def add_pad_token(self, token=PAD) -> ITokenizer:
        token_id = self.add_token(token)
        self.special_tokens._pad = (token, token_id)
        return self

    @check_token(BLANK)
    def add_blank_token(self, token=BLANK) -> ITokenizer:
        token_id = self.add_token(token)
        self.special_tokens._blank = (token, token_id)
        return self

    @check_token(SOS)
    def add_sos_token(self, token=SOS) -> ITokenizer:
        token_id = self.add_token(token)
        self.special_tokens._sos = (token, token_id)
        return self

    @check_token(EOS)
    def add_eos_token(self, token=EOS) -> ITokenizer:
        token_id = self.add_token(token)
        self.special_tokens._eos = (token, token_id)
        return self

    def _reset_id_to_token(self) -> None:
        self._id_to_token = dict(zip(
            self._token_to_id.values(),
            self._token_to_id.keys()
            ))

    def __set_special_tokens_dict(self, data: dict) -> None:
        if self._pad_key in data:
            self.special_tokens._pad = tuple(data[self._pad_key])
        if self._blank_key in data:
            self.special_tokens._blank = tuple(data[self._blank_key])
        if self._sos_key in data:
            self.special_tokens._sos = tuple(data[self._sos_key])
        if self._eos_key in data:
            self.special_tokens._eos = tuple(data[self._eos_key])

    def __get_special_tokens_dict(self) -> dict:
        data = {}
        if self.special_tokens.pad_id is not None:
            data[self._pad_key] = list(self.special_tokens._pad)
        if self.special_tokens.blank_id is not None:
            data[self._blank_key] = list(self.special_tokens._blank)
        if self.special_tokens.sos_id is not None:
            data[self._sos_key] = list(self.special_tokens._sos)
        if self.special_tokens.eos_id is not None:
            data[self._eos_key] = list(self.special_tokens._eos)
        return data

    def load_tokenizer(
            self,
            tokenizer_path: Union[str, PathLike],
            *args,
            **kwargs
            ) -> ITokenizer:
        data = JSONLoader(tokenizer_path).load()
        self._token_to_id = data[self._token_to_id_key]
        self.__set_special_tokens_dict(data[self._special_tokens_key])
        self._reset_id_to_token()
        return self

    def set_tokenizer(self, data: List[str], *args, **kwargs) -> ITokenizer:
        all_tokens = self.get_tokens(data)
        _ = list(map(self.add_token, all_tokens))
        self._reset_id_to_token()
        return self

    def save_tokenizer(
            self,
            save_path: Union[str, PathLike],
            *args,
            **kwargs
            ) -> None:
        data = {
            self._token_to_id_key: self._token_to_id,
            self._special_tokens_key: self.__get_special_tokens_dict()
        }
        save_json(save_path, data)

    def ids2tokens(self, ids: List[str]) -> List[str]:
        return list(map(lambda x: self._id_to_token[x], ids))

    def tokens2ids(self, sentence: str) -> List[int]:
        sentence = self.preprocess_tokens(sentence)
        return list(map(
            lambda x: self._token_to_id.get(x, self.special_tokens.pad_id),
            sentence)
            )

    def batch_tokenizer(self, data: List[str]) -> list:
        return list(map(self.tokens2ids, data))

    def batch_detokenizer(self, data: List[int]) -> list:
        return list(map(self.ids2tokens, data))


class CharTokenizer(BaseTokenizer):
    def __init__(self) -> None:
        super().__init__()

    def get_tokens(self, data: List[str]):
        return set(''.join(data))

    def preprocess_tokens(self, sentence: str) -> List[str]:
        return list(sentence)

# dataset.py

In [ ]:
import math
import pandas as pd
import torch
from tokenizer import ITokenizer
from utils import IPipeline
from pathlib import Path
from typing import List, Union
from torch import Tensor


class BaseData:
    def __init__(
            self,
            text_pipeline: IPipeline,
            audio_pipeline: IPipeline,
            tokenizer: ITokenizer,
            sampling_rate: int,
            hop_length: int,
            fields_sep: str,
            csv_file_keys: object
            ) -> None:
        self.text_pipeline = text_pipeline
        self.audio_pipeline = audio_pipeline
        self.tokenizer = tokenizer
        self.max_len = 0
        self.sampling_rate = sampling_rate
        self.hop_length = hop_length
        self.sep = fields_sep
        self.csv_file_keys = csv_file_keys

    def _get_padded_aud(
            self,
            aud_path: Union[str, Path],
            max_duration: int,
            ) -> Tensor:
        max_len = 1 + math.ceil(
            max_duration * self.sampling_rate / self.hop_length
            )
        # Updates teh max_len to be used for text padding
        self.max_len = max_len
        aud = self.audio_pipeline.run(aud_path)
        assert aud.shape[0] == 1, f'expected audio of 1 channels got \
            {aud_path} with {aud.shape[0]} channels'
        return self.pad_mels(aud, max_len)

    def _get_padded_tokens(self, text: str) -> Tensor:
        text = self.text_pipeline.run(text)
        tokens = self.tokenizer.tokens2ids(text)
        length = len(tokens)
        tokens = self.pad_tokens(tokens)
        return torch.LongTensor(tokens), length

    def prepocess_lines(self, data: str) -> List[str]:
        return [
            item.split(self.sep)
            for item in data
        ]

    def pad_mels(self, mels: Tensor, max_len: int) -> Tensor:
        n = max_len - mels.shape[1]
        zeros = torch.zeros(size=(1, n, mels.shape[-1]))
        return torch.cat([zeros, mels], dim=1)

    def pad_tokens(self, tokens: list) -> Tensor:
        length = self.max_len - len(tokens)
        return tokens + [self.tokenizer.special_tokens.pad_id] * length


class DataLoader(BaseData):
    def __init__(
            self,
            file_path: Union[str, Path],
            text_pipeline: IPipeline,
            audio_pipeline: IPipeline,
            tokenizer: ITokenizer,
            batch_size: int,
            sampling_rate: int,
            hop_length: int,
            fields_sep: str,
            csv_file_keys: object
            ) -> None:
        super().__init__(
                text_pipeline,
                audio_pipeline,
                tokenizer,
                sampling_rate,
                hop_length,
                fields_sep,
                csv_file_keys
                )
        self.batch_size = batch_size
        self.df = pd.read_csv(file_path)
        self.num_examples = len(self.df)
        self.idx = 0

    def __len__(self):
        length = self.num_examples // self.batch_size
        mod = self.num_examples % self.batch_size
        return length + 1 if mod > 0 else length

    def get_max_duration(self, start_idx: int, end_idx: int) -> float:
        return self.df[
            self.csv_file_keys.duration
            ].iloc[start_idx: end_idx].max()

    def get_audios(self, start_idx: int, end_idx: int) -> Tensor:
        max_duration = self.get_max_duration(start_idx, end_idx)
        result = list(map(
            self._get_padded_aud,
            self.df[self.csv_file_keys.path].iloc[start_idx: end_idx],
            [max_duration] * (end_idx - start_idx)
            ))
        result = torch.stack(result, dim=1)
        return torch.squeeze(result)

    def get_texts(self, start_idx: int, end_idx: int) -> Tensor:
        args = self.df[self.csv_file_keys.text].iloc[start_idx: end_idx]
        result = list(map(self._get_padded_tokens, args))
        length = list(map(lambda x: x[1], result))
        result = list(map(lambda x: x[0], result))
        result = torch.stack(result, dim=0)
        return result, torch.LongTensor(length)

    def __iter__(self):
        self.idx = 0
        while True:
            start = self.idx * self.batch_size
            end = (self.idx + 1) * self.batch_size
            end = min(end, self.num_examples)
            if start > self.num_examples or start == end:
                break
            self.idx += 1
            yield (
                self.get_audios(start, end),
                self.get_texts(start, end)
                )

# dataloader.py

In [ ]:
from abc import ABC, abstractmethod
from typing import  Union
from os import PathLike
import json


class IDataLoader(ABC):

    @abstractmethod
    def load(self):
        pass

class JSONLoader(IDataLoader):
    def __init__(self, file_path: Union[str, PathLike]) -> None:
        super().__init__()
        self.file_path = file_path

    def load(self):
        with open(self.file_path, 'r') as f:
            data = json.load(f)
        return data

class TextLoader(IDataLoader):
    def __init__(self, file_path: Union[str, PathLike]) -> None:
        super().__init__()
        self.file_path = file_path

    def load(self):
        with open(self.file_path, 'r') as f:
            data = f.read()
        return data

# collate.py

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence

def speech_collate_fn(batch,pad_idx=0):
  waveforms,labels=zip(*batch)
  waveform_lengths=torch.tensor([w.shape[0] for w in waveforms],dtype=torch.long)
  label_lengths=torch.tensor([len(l) for l in labels],dtype=torch.long)

  padded_waveforms=pad_sequence(waveforms,batch_first=True)
  padded_labels=pad_sequence(labels,batch_first=True,padding_value=pad_idx)

  return padded_waveforms,waveform_lengths,padded_labels,label_lengths

# hparams.py

In [ ]:
hparams = {
    'num_classes': 30,  # a~z + space + apostrophe + blank 등 (수정 가능)
    'input_dim': 80,    # log-mel feature 수

    'encoder_dim': 512,  # model_dim과 동일
    'num_encoder_layers': 4,  # enc.num_blocks
    'num_attention_heads': 8,  # model.enc.mhsa_params.h
    'feed_forward_expansion_factor': 2,  # model.enc.feed_forward_params.scaling_factor
    'conv_expansion_factor': 2,          # model.enc.conv_mod_params.scaling_factor

    'input_dropout_p': 0.1,  # model.p_dropout
    'feed_forward_dropout_p': 0.1,  # model.enc.feed_forward_params.p_dropout
    'attention_dropout_p': 0.1,     # model.enc.mhsa_params.p_dropout
    'conv_dropout_p': 0.1,          # model.enc.conv_mod_params.p_dropout

    'conv_kernel_size': 31,         # model.enc.conv_mod_params.kernel_size
    'half_step_residual': True,     # model.enc.feed_forward_params.residual_scaler = 0.5 → True로 해석

    'sampling_rate': 16000,
    'n_mels': 80,
    'n_fft': 400,
    'hop_length': 160,
    'win_length': 400,

    'spec_aug': {
        'freq_mask': {
            'max_freq': 27
        },
        'time_mask': {
            'ps': 0.05,
            'number_of_masks': 10
        }
    },

    'train_csv': 'files/train.csv',# 경로
    'test_csv': 'files/test.csv',
    'csv_file_keys': {
        'duration': 'duration',
        'path': 'path',
        'text': 'text'
    },

    'batch_size': 16,
    'epochs': 100,

    'optimizer': {
        'beta1': 0.9,
        'beta2': 0.98,
        'eps': 1e-9,
        'weight_decay': 1e-6,
        'warmup_steps': int(1e4),
        'scaler': 0.05,
        'step_size': 1
    }
}


#  model.py

In [ ]:
import math
from typing import List
import torch
import torch.nn as nn
from torch import Tensor
from functools import lru_cache


class ConformerBlock(nn.Module):
    def __init__(
            self,
            enc_dim: int,
            mhsa_params: dict,
            conv_module_params: dict,
            ff_module_params: dict
            ) -> None:
        super().__init__()
        self.ff1 = FeedForwardModule(**ff_module_params)
        self.mhsa = MHSA(**mhsa_params)
        self.conv = ConvModule(**conv_module_params)
        self.ff2 = FeedForwardModule(**ff_module_params)
        self.lnorm = nn.LayerNorm(enc_dim)

    def forward(self, inp: Tensor):
        out = self.ff1(inp)
        out = self.mhsa(out)
        out = self.conv(out)
        out = self.ff2(out)
        out = self.lnorm(out)
        return out


class MHSA(nn.Module):
    def __init__(
            self,
            enc_dim: int,
            h: int,
            p_dropout: float,
            device: str
            ) -> None:
        super().__init__()
        assert enc_dim % h == 0, 'enc_dim is not divisible by h'
        self.fc_key = nn.Linear(
            in_features=enc_dim,
            out_features=enc_dim,
        )
        self.fc_query = nn.Linear(
            in_features=enc_dim,
            out_features=enc_dim,
        )
        self.fc_value = nn.Linear(
            in_features=enc_dim,
            out_features=enc_dim,
        )
        self.proj_fc = nn.Linear(
            in_features=2 * enc_dim,
            out_features=enc_dim,
        )
        self.lnorm = nn.LayerNorm(enc_dim)
        self.dropout = nn.Dropout(p_dropout)
        self.enc_dim = enc_dim
        self.h = h
        self.dk = enc_dim // h
        self.sqrt_dk = math.sqrt(self.dk)
        self.softmax = nn.Softmax(dim=-1)
        self.device = device

    def _get_scaled_att(
            self,
            Q: Tensor,
            K: Tensor
            ) -> Tensor:
        """Calculates the scaled attention map
        by calculating softmax(matmul(Q, K.T)/sqrt(dk))
        Args:
            Q (Tensor): The Query tensor of shape [h * B, Tq, dk]
            K (Tensor): The Key tensor of shape [h * B, dk, Tk]
        Returns:
            Tensor: The scaled attention weights of shape
            [B * h, Tq, Tk]
        """
        result = torch.matmul(Q, K)
        result = result / self.sqrt_dk
        return self.softmax(result)

    def perform_att(
            self,
            Q: Tensor,
            K: Tensor,
            V: Tensor
            ) -> Tensor:
        """Performs multi-head scaled attention
        by calculating softmax(matmul(Q, K.T)/sqrt(dk)).V
        Args:
            Q (Tensor): The Query tensor of shape [h * B, Tq, dk]
            K (Tensor): The Key tensor of shape [h * B, dk, Tk]
            V (Tensor): The Value tensor of shape [h * B, Tk, dk]
        Returns:
            Tuple[Tensor, Tensor]: The attention matrix of shape
            [B * h, Tq, Tk] and the scaled attention value of
            shape [B * h, Tq, dk].
        """
        att = self._get_scaled_att(Q, K)
        result = torch.matmul(att, V)
        return att, result

    @lru_cache(maxsize=2)
    def get_positionals(self, max_length: int) -> Tensor:
        """Create Positionals tensor to be added to the input
        Args:
            max_length (int): The maximum length of the positionals sequence.
        Returns:
            Tensor: Positional tensor
        """
        result = torch.zeros(max_length, self.enc_dim, dtype=torch.float)
        for pos in range(max_length):
            for i in range(0, self.enc_dim, 2):
                denominator = pow(10000, 2 * i / self.enc_dim)
                result[pos, i] = math.sin(pos / denominator)
                result[pos, i + 1] = math.cos(pos / denominator)
        return result

    def _reshape(self, *args) -> List[Tensor]:
        """Reshabes all the given list of tensor
        from [B, T, N] to [B, T, h, dk]
        Returns:
            List[Tensor]: list of all reshaped tensors
        """
        return [
            item.contiguous().view(-1, item.shape[1], self.h, self.dk)
            for item in args
        ]

    def _pre_permute(self, *args) -> List[Tensor]:
        """Permutes all the given list of tensors
        from [B, T, h, dk] to become [h, B, T, dk].

        Returns:
            List[Tensor]: List of all permuted tensors.
        """
        return [
            item.permute(2, 0, 1, 3)
            for item in args
        ]

    def _change_dim(self, *args) -> List[Tensor]:
        """Changes the dimensionality of all passed tensores
        from [B, T, N] to [B * h, T, dk]

        Returns:
            List[Tensor]: List of the modified tensors.
        """
        result = self._reshape(*args)  # [B, T, h, dk]
        result = self._pre_permute(*result)  # [h, B, T, dk]
        return [
            item.permute(1, 0, 2, 3).contiguous().view(
                -1, item.shape[2], item.shape[3]
                )
            for item in result
        ]

    def forward(self, inp: Tensor) -> Tensor:
        """Passes the input into multi-head attention

        Args:
            inp (Tensor): The input tensor

        Returns:
            Tensor: The result after adding it to positionals
            and passing it through multi-head self-attention
        """
        out = self.lnorm(inp)
        [b, s, _] = inp.shape
        K = self.fc_key(inp)
        Q = self.fc_query(inp)
        V = self.fc_value(inp)
        (Q, K, V) = self._change_dim(Q, K, V)  # [h * B, T, dk]
        K = K.permute(0, 2, 1)  # [h, T, B, dk]
        _, result = self.perform_att(Q, K, V)
        result = result.view(b, self.h, s, self.dk)
        result = result.permute(0, 2, 1, 3)
        result = result.contiguous().view(b, s, -1)
        result = torch.cat([inp, result], dim=-1)
        result = self.proj_fc(result)
        out = self.dropout(result)
        return inp + out


class ConvModule(nn.Module):
    """Implements the convolution module
    where it contains the following layers
    1. Layernorm
    2. Pointwise Conv
    3. Gate Linear unit
    4. 1D Depthwise conv
    5. BatchNorm
    6. Swish Activation
    7. Pointwise Conv
    8. Dropout

    Args:
        enc_dim (int): The encoder dimensionality.
        scaling_factor (int): The scaling factor of the conv layer.
        kernel_size (int): The convolution kernel size.
        p_dropout (float): The dropout probability.
    """
    def __init__(
            self,
            enc_dim: int,
            scaling_factor: int,
            kernel_size: int,
            p_dropout: float
            ) -> None:
        super().__init__()
        self.lnorm = nn.LayerNorm(enc_dim)
        n_scaled_channels = enc_dim * scaling_factor
        assert (kernel_size - 1) % 2 == 0, 'kernel_size - 1 \
            must be divisable by 2 -odd'
        padding_size = (kernel_size - 1) // 2
        self.pwise_conv1 = nn.Conv1d(
            in_channels=enc_dim,
            out_channels=n_scaled_channels,
            kernel_size=1
        )
        self.glu = nn.GLU(dim=1)
        self.dwise_conv = nn.Conv1d(
            in_channels=enc_dim,
            out_channels=enc_dim,
            kernel_size=kernel_size,
            padding=padding_size,
            groups=enc_dim
        )
        self.bnorm = nn.BatchNorm1d(enc_dim)
        self.swish = nn.SiLU()
        self.dropout = nn.Dropout(p_dropout)
        self.pwise_conv2 = nn.Conv1d(
            in_channels=enc_dim,
            out_channels=enc_dim,
            kernel_size=1
        )

    def forward(self, inp: Tensor) -> Tensor:
        out = self.lnorm(inp)
        out = out.permute(0, 2, 1)
        out = self.pwise_conv1(out)
        out = self.glu(out)
        out = self.dwise_conv(out)
        out = self.bnorm(out)
        out = self.swish(out)
        out = self.pwise_conv2(out)
        out = self.dropout(out)
        out = out.permute(0, 2, 1)
        return out + inp


class FeedForwardModule(nn.Module):
    """Implements the feed forward module in the conformer block
    where the module consists of the below
    1. Layer Norm
    2. Linear Layer
    3. Swish Activation
    4. Dropout
    5. Linear Layer
    6. Dropout

    Args:
        enc_dim (int): The encoder dimensionality
        scaling_factor (int): The scaling factor of the linear layer
        p_dropout (float): The dropout probability
        residual_scaler (float, optional): The residual scaling.
        Defaults to 0.5.
    """
    def __init__(
            self,
            enc_dim: int,
            scaling_factor: int,
            p_dropout: float,
            residual_scaler=0.5
            ) -> None:
        super().__init__()
        self.residual_scaler = residual_scaler
        scaled_dim = scaling_factor * enc_dim
        self.lnorm = nn.LayerNorm(enc_dim)
        self.fc1 = nn.Linear(
            in_features=enc_dim,
            out_features=scaled_dim
        )
        self.fc2 = nn.Linear(
            in_features=scaled_dim,
            out_features=enc_dim
        )
        self.swish = nn.SiLU()
        self.dropout = nn.Dropout(p_dropout)

    def forward(self, inp: Tensor, x=4) -> Tensor:
        """Passes the given inp through the feed forward
        module

        Args:
            inp (Tensor): the input to the feed forward module
            with shape [B, M, N] where B is the batch size, M
            is the maximum length, and N is the encoder dim
        Returns:
            Tensor: The result of the forward module
        """
        out = self.lnorm(inp)
        out = self.fc1(out)
        out = self.swish(out)
        out = self.dropout(out)
        out = self.fc2(out)
        out = self.dropout(out)
        return self.residual_scaler * inp + out


class Encoder(nn.Module):
    def __init__(
            self,
            enc_dim: int,
            in_channels: int,
            kernel_size: int,
            out_channels: int,
            mhsa_params: dict,
            num_blocks: int,
            p_dropout: float,
            conv_mod_params: dict,
            feed_forward_params: dict
            ) -> None:
        super().__init__()
        self.subsampling_conv = nn.Conv1d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size
        )
        self.fc = nn.Linear(
            in_features=out_channels,
            out_features=enc_dim
        )
        self.dropout = nn.Dropout(p_dropout)
        self.conformers_layers = nn.ModuleList([
            ConformerBlock(
                enc_dim,
                mhsa_params,
                conv_mod_params,
                feed_forward_params
                )
            for _ in range(num_blocks)
        ])

    def forward(self, inp: Tensor):
        inp = inp.permute(0, 2, 1)
        out = self.subsampling_conv(inp)
        out = out.permute(0, 2, 1)
        out = self.fc(out)
        out = self.dropout(out)
        for layer in self.conformers_layers:
            out = layer(out)
        return out


class Decoder(nn.Module):
    def __init__(
            self,
            enc_dim: int,
            vocab_size: int
            ) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=enc_dim,
            hidden_size=vocab_size,
            batch_first=True
        )

    def forward(self, inp: Tensor) -> Tensor:
        output, *_ = self.lstm(inp)
        return output


class Model(nn.Module):
    def __init__(
            self,
            enc_params: dict,
            dec_params: dict
            ) -> None:
        super().__init__()
        self.encoder = Encoder(**enc_params)
        self.decoder = Decoder(**dec_params)

    def forward(self, inp: Tensor) -> Tensor:
        return self.decoder(self.encoder(inp))

# train.py

In [ ]:
import os
from pathlib import Path
import torch
from hprams import (
    get_audpipe_params,
    get_model_params,
    get_optim_params,
    hprams
    )
from functools import wraps
from torch.nn import Module
#from data import DataLoader
from torch.optim import Optimizer
from typing import Callable, Union
from tqdm import tqdm
from model import Model

from optim import AdamWarmup
from pipelines import get_pipelines
from tokenizer import CharTokenizer, ITokenizer
from utils import IPipeline

MODEL_KEY = 'model'
OPTIMIZER_KEY = 'optimizer'


def save_checkpoint(func) -> Callable:
    """Save a checkpoint after each iteration
    """
    @wraps(func)
    def wrapper(obj, *args, _counter=[0], **kwargs):
        _counter[0] += 1
        result = func(obj, *args, **kwargs)
        if not os.path.exists(hprams.training.checkpoints_dir):
            os.mkdir(hprams.training.checkpoints_dir)
        checkpoint_path = os.path.join(
            hprams.training.checkpoints_dir,
            'checkpoint_' + str(_counter[0]) + '.pt'
            )
        torch.save(
            {
                MODEL_KEY: obj.model.state_dict(),
                OPTIMIZER_KEY: obj.optimizer.state_dict(),
            },
            checkpoint_path
            )
        print(f'checkpoint saved to {checkpoint_path}')
        return result
    return wrapper


class Trainer:
    __train_loss_key = 'train_loss'
    __test_loss_key = 'test_loss'

    def __init__(
            self,
            criterion: Module,
            optimizer: Optimizer,
            model: Module,
            device: str,
            train_loader: DataLoader,
            test_loader: DataLoader,
            epochs: int
            ) -> None:
        self.criterion = criterion
        self.optimizer = optimizer
        self.model = model
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.device = device
        self.epochs = epochs
        self.step_history = dict()
        self.history = dict()

    def fit(self):
        """The main training loop that train the model on the training
        data then test it on the test set and then log the results
        """
        for _ in range(self.epochs):
            self.train()
            self.test()
            self.print_results()

    def set_train_mode(self) -> None:
        """Set the models on the training mood
        """
        self.model = self.model.train()

    def set_test_mode(self) -> None:
        """Set the models on the testing mood
        """
        self.model = self.model.eval()

    def print_results(self):
        """Prints the results after each epoch
        """
        result = ''
        for key, value in self.history.items():
            result += f'{key}: {str(value[-1])}, '
        print(result[:-2])

    def test(self):
        """Iterate over the whole test data and test the models
        for a single epoch
        """
        total_loss = 0
        self.set_test_mode()
        for x, (y, target_lengths) in tqdm(self.test_loader):
            x = x.to(self.device)
            y = y.to(self.device)
            result = self.model(x)
            result = result.log_softmax(axis=-1)
            result = result.permute(1, 0, 2)
            y = y[..., :result.shape[0]]
            input_lengths = torch.full(
                size=(y.shape[0],),
                fill_value=y.shape[1],
                dtype=torch.long
                )
            loss = self.criterion(
                result,
                y,
                input_lengths,
                target_lengths
            )
            total_loss += loss.item()
        total_loss /= len(self.train_loader)
        if self.__test_loss_key in self.history:
            self.history[self.__test_loss_key].append(total_loss)
        else:
            self.history[self.__test_loss_key] = [total_loss]

    @save_checkpoint
    def train(self):
        """Iterates over the whole training data and train the models
        for a single epoch
        """
        torch.autograd.set_detect_anomaly(True)
        total_loss = 0
        self.set_train_mode()
        for x, (y, target_lengths) in tqdm(self.train_loader):
            x = x.to(self.device)
            y = y.to(self.device)
            self.optimizer.zero_grad()
            result = self.model(x)
            result = result.log_softmax(axis=-1)
            result = result.permute(1, 0, 2)
            y = y[..., :result.shape[0]]
            input_lengths = torch.full(
                size=(y.shape[0],),
                fill_value=y.shape[1],
                dtype=torch.long
                )
            loss = self.criterion(
                result,
                y,
                input_lengths,
                target_lengths
            )
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        total_loss /= len(self.train_loader)
        if self.__train_loss_key in self.history:
            self.history[self.__train_loss_key].append(total_loss)
        else:
            self.history[self.__train_loss_key] = [total_loss]


def get_criterion(blank_id: int) -> Module:
    return torch.nn.CTCLoss(blank_id)


def get_optimizer(model: Module, params: dict) -> object:
    return AdamWarmup(
        model.parameters(),
        **params
        )


def load_model(
        model_params: dict,
        checkpoint_path=None
        ) -> Module:
    model = Model(**model_params)
    if checkpoint_path is not None:
        model.load_state_dict(
            torch.load(checkpoint_path)[MODEL_KEY]
            )
    return model


def get_data_loader(
        file_path: Union[str, Path],
        tokenizer: ITokenizer,
        text_pipeline: IPipeline,
        audio_pipeline: IPipeline,
        ):
    return DataLoader(
        file_path,
        text_pipeline,
        audio_pipeline,
        tokenizer,
        hprams.training.batch_size,
        hprams.data.sampling_rate,
        hprams.data.hop_length,
        hprams.data.files_sep,
        hprams.data.csv_file_keys,
    )


def get_tokenizer():
    tokenizer = CharTokenizer()
    if hprams.tokenizer.tokenizer_file is not None:
        tokenizer = tokenizer.load_tokenizer(
            hprams.tokenizer.tokenizer_file
            )
    tokenizer = tokenizer.add_blank_token().add_pad_token()
    with open(hprams.tokenizer.vocab_path, 'r') as f:
        vocab = f.read().split('\n')
    tokenizer.set_tokenizer(vocab)
    tokenizer.save_tokenizer('tokenizer.json')
    return tokenizer


def get_train_test_loaders(
        tokenizer: ITokenizer,
        audio_pipeline: IPipeline,
        text_pipeline: IPipeline
        ) -> tuple:
    return (
        get_data_loader(
            hprams.data.training_file,
            tokenizer,
            text_pipeline,
            audio_pipeline
            ),
        get_data_loader(
            hprams.data.testing_file,
            tokenizer,
            text_pipeline,
            audio_pipeline
            )
    )


def get_trainer() -> Trainer:
    device = hprams.device
    tokenizer = get_tokenizer()
    blank_id = tokenizer.special_tokens.blank_id
    vocab_size = tokenizer.vocab_size
    text_pipeline, audio_pipeline = get_pipelines(
        get_audpipe_params()
    )
    model = load_model(
        get_model_params(vocab_size),
        checkpoint_path=hprams.checkpoint
        ).to(device)
    train_loader, test_loader = get_train_test_loaders(
        tokenizer,
        audio_pipeline,
        text_pipeline
    )
    return Trainer(
        criterion=get_criterion(blank_id),
        optimizer=get_optimizer(
            model, get_optim_params()
            ),
        model=model,
        device=device,
        train_loader=train_loader,
        test_loader=test_loader,
        epochs=hprams.training.epochs
    )


if __name__ == '__main__':
    trainer = get_trainer()
    trainer.fit()